# Abalone Age Prediction: Modelling with MLflow Experiment Tracking

The goal of this notebook is to implement, evaluate, and iteratively improve a machine learning model to predict the age (number of **`Rings`**) of an abalone from physical measurements.

### Workflow & Experiments
1. **Data Preprocessing Pipeline**:
   - Clean anomalous values identified during EDA (`Height <= 0`, extreme height outliers, and weight typos).
   - Train/Test Split (80/20, `random_state=42`) with Scikit-Learn `ColumnTransformer` (One-Hot Encoding for `Sex`, `StandardScaler` for numeric features).
2. **Phase 1: Baseline Models**:
   - **Baseline 0**: `DummyRegressor` (mean predictor lower bound).
   - **Model 1**: `LinearRegression` (interpretable baseline, per project instructions).
   - **Model 2**: `Ridge` ($L_2$ regularized linear model).
   - **Model 3**: `RandomForestRegressor` (non-linear ensemble).
3. **High-Rings Bias Diagnosis**:
   - Quantify underprediction error across age brackets (Young $\le 8$, Prime $9-12$, Older $13-15$, Oldest $> 15$).
4. **Phase 2: Strategy 1 – Domain Feature Engineering**:
   - `shell_ratio`: Shell weight / Whole weight (captures continuous shell calcification as age increases).
   - `meat_ratio`: Shucked weight / Whole weight.
   - `volume`: Length * Diameter * Height (3D volumetric physical estimate).
5. **Phase 3: Strategy 2 – Cost-Sensitive Sample Weighting**:
   - Weight specimens with `Rings > 12` higher during training to mitigate class imbalance and pull predictions up for older abalones.
6. **Comparison & MLflow Tracking**:
   - Query all runs logged to local MLflow experiment `abalone-age-prediction` and compare RMSE, MAE, $R^2$, and high-ring bias.

## 1. Imports and Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

## 2. MLflow Tracking Setup
We configure a local MLflow experiment to log all model runs.

In [ ]:
# Allow filesystem store in MLflow 3.x and disable hint logs
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

# Set local MLflow tracking directory
tracking_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "mlruns"))
mlflow.set_tracking_uri(f"file://{tracking_dir}")

experiment_name = "abalone-age-prediction"
mlflow.set_experiment(experiment_name)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {experiment_name}")

## 3. Data Loading and Cleaning
Based on our exploratory data analysis:
- Filter out records where `Height <= 0` or `Height >= 0.5` (physical measurement anomalies).
- Filter out records where `Shucked weight > Whole weight` or `Shell weight > Whole weight` (data entry errors where a component exceeds the whole animal).

In [ ]:
data_path = os.path.join("..", "data", "abalone.csv")
df = pd.read_csv(data_path)

initial_rows = len(df)
df_clean = (
    df[
        (df["Height"] > 0)
        & (df["Height"] < 0.5)
        & (df["Shucked weight"] <= df["Whole weight"])
        & (df["Shell weight"] <= df["Whole weight"])
    ]
    .copy()
    .reset_index(drop=True)
)

dropped_rows = initial_rows - len(df_clean)
print(f"Loaded {initial_rows} records. Removed {dropped_rows} anomalous records ({len(df_clean)} remaining).")

## 4. Evaluation & MLflow Logging Helper
We define a reusable function to fit any model, compute comprehensive metrics (overall RMSE/MAE/R² and high-ring bias on `Rings > 12` and `Rings > 15`), and log everything to MLflow.

In [ ]:
def evaluate_and_log_run(
    run_name: str,
    pipeline: Pipeline,
    X_train_data: pd.DataFrame,
    y_train_data: pd.Series,
    X_test_data: pd.DataFrame,
    y_test_data: pd.Series,
    params: dict = None,
    fit_params: dict = None,
):
    """Train a model pipeline, evaluate metrics, and log artifacts to MLflow."""
    if params is None:
        params = {}
    if fit_params is None:
        fit_params = {}

    with mlflow.start_run(run_name=run_name):
        # 1. Fit
        pipeline.fit(X_train_data, y_train_data, **fit_params)

        # 2. Predict
        y_train_pred = pipeline.predict(X_train_data)
        y_test_pred = pipeline.predict(X_test_data)

        # 3. Overall Metrics
        train_rmse = root_mean_squared_error(y_train_data, y_train_pred)
        test_rmse = root_mean_squared_error(y_test_data, y_test_pred)
        train_mae = mean_absolute_error(y_train_data, y_train_pred)
        test_mae = mean_absolute_error(y_test_data, y_test_pred)
        train_r2 = r2_score(y_train_data, y_train_pred)
        test_r2 = r2_score(y_test_data, y_test_pred)

        # 4. High-Rings Bias (Predicted - Actual)
        mask_12 = y_test_data > 12
        mask_15 = y_test_data > 15
        bias_gt_12 = float((y_test_pred[mask_12] - y_test_data[mask_12]).mean())
        bias_gt_15 = float((y_test_pred[mask_15] - y_test_data[mask_15]).mean())

        # 5. Log Parameters & Metrics
        mlflow.log_param("model_name", run_name)
        for k, v in params.items():
            mlflow.log_param(k, v)

        mlflow.log_metric("train_rmse", train_rmse)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("test_mae", test_mae)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_r2", test_r2)
        mlflow.log_metric("bias_gt_12", bias_gt_12)
        mlflow.log_metric("bias_gt_15", bias_gt_15)

        # 6. Log model artifact
        mlflow.sklearn.log_model(pipeline, artifact_path="model", serialization_format="cloudpickle")

        print(f"=== {run_name} ===")
        print(f"Test RMSE: {test_rmse:.3f} | Test MAE: {test_mae:.3f} | Test R2: {test_r2:.3f}")
        print(f"Bias (Rings > 12): {bias_gt_12:.2f} | Bias (Rings > 15): {bias_gt_15:.2f}\n")

        return {
            "run_name": run_name,
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "train_mae": train_mae,
            "test_mae": test_mae,
            "train_r2": train_r2,
            "test_r2": test_r2,
            "bias_gt_12": bias_gt_12,
            "bias_gt_15": bias_gt_15,
            "pipeline": pipeline,
            "y_test_pred": y_test_pred,
        }

## 5. Phase 1: Baseline Models (Raw Features)
We start with the raw features using an 80/20 train/test split.

In [ ]:
target_col = "Rings"
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ["Sex"]
num_cols = [c for c in X.columns if c not in cat_cols]

raw_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ]
)

all_results = []

# 1. Dummy Baseline
pipe_dummy = Pipeline([("prep", raw_preprocessor), ("reg", DummyRegressor(strategy="mean"))])
all_results.append(
    evaluate_and_log_run("1_Dummy_Baseline", pipe_dummy, X_train, y_train, X_test, y_test, {"strategy": "mean"})
)

# 2. Linear Regression Baseline
pipe_lr = Pipeline([("prep", raw_preprocessor), ("reg", LinearRegression())])
all_results.append(
    evaluate_and_log_run(
        "2_Linear_Regression_Base", pipe_lr, X_train, y_train, X_test, y_test, {"model_type": "LinearRegression"}
    )
)

# 3. Ridge Regression Baseline
pipe_ridge = Pipeline([("prep", raw_preprocessor), ("reg", Ridge(alpha=1.0, random_state=42))])
all_results.append(evaluate_and_log_run("3_Ridge_Base", pipe_ridge, X_train, y_train, X_test, y_test, {"alpha": 1.0}))

# 4. Random Forest Baseline
pipe_rf = Pipeline(
    [
        ("prep", raw_preprocessor),
        ("reg", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ]
)
all_results.append(
    evaluate_and_log_run(
        "4_Random_Forest_Base", pipe_rf, X_train, y_train, X_test, y_test, {"n_estimators": 100, "max_depth": 10}
    )
)

## 6. Diagnosis: Quantifying High-Rings Bias
Let's inspect the residuals across age brackets for the baseline models.

In [ ]:
res_df = pd.DataFrame(
    {"Actual": y_test, "Pred_LR": all_results[1]["y_test_pred"], "Pred_RF": all_results[3]["y_test_pred"]}
)
res_df["Residual_LR"] = res_df["Pred_LR"] - res_df["Actual"]
res_df["Residual_RF"] = res_df["Pred_RF"] - res_df["Actual"]

bins = [0, 8, 12, 15, 30]
labels = ["Young (1-8)", "Prime (9-12)", "Older (13-15)", "Oldest (>15)"]
res_df["Ring_Group"] = pd.cut(res_df["Actual"], bins=bins, labels=labels)

bias_breakdown = res_df.groupby("Ring_Group", observed=False)[["Residual_LR", "Residual_RF"]].agg(["count", "mean"])
print("Mean Bias (Predicted - Actual): Negative indicates underprediction")
bias_breakdown

**Diagnosis Findings**:
1. **Young specimens (1-8 rings)**: Models slightly overpredict (+0.8 to +1.0 rings).
2. **Prime adults (9-12 rings)**: Models are well calibrated (+0.2 rings).
3. **Older specimens (13-15 rings)**: Models underpredict by **~1.7 to 1.8 rings**.
4. **Oldest specimens (>15 rings)**: Severe underprediction by **~4.5 rings**.

**Why?**
- **Biological Growth Plateau**: Physical dimensions flatten after maturity while age rings continue accumulating.
- **Class Imbalance**: >80% of data has $\le 12$ rings, so standard MSE minimizes aggregate loss by shrinking predictions toward the global mean (~9.9).

## 7. Strategy 1: Domain Feature Engineering
To help the models distinguish older abalones whose overall size has plateaued:
1. **`shell_ratio`** (`Shell weight / Whole weight`): Shell calcification continues throughout life even after flesh stops growing.
2. **`meat_ratio`** (`Shucked weight / Whole weight`): The proportion of soft body mass relative to whole animal.
3. **`volume`** (`Length * Diameter * Height`): An approximation of 3D volume.

In [ ]:
df_fe = df_clean.copy()
df_fe["shell_ratio"] = df_fe["Shell weight"] / df_fe["Whole weight"]
df_fe["meat_ratio"] = df_fe["Shucked weight"] / df_fe["Whole weight"]
df_fe["volume"] = df_fe["Length"] * df_fe["Diameter"] * df_fe["Height"]

X_fe = df_fe.drop(columns=[target_col])
y_fe = df_fe[target_col]

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, test_size=0.2, random_state=42)

cat_cols_fe = ["Sex"]
num_cols_fe = [c for c in X_fe.columns if c not in cat_cols_fe]

fe_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_fe),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols_fe),
    ]
)

# Linear Regression + Feature Engineering
pipe_lr_fe = Pipeline([("prep", fe_preprocessor), ("reg", LinearRegression())])
all_results.append(
    evaluate_and_log_run(
        "5_Linear_Regression_FE",
        pipe_lr_fe,
        X_train_fe,
        y_train_fe,
        X_test_fe,
        y_test_fe,
        {"features": "domain_engineered"},
    )
)

# Ridge + Feature Engineering
pipe_ridge_fe = Pipeline([("prep", fe_preprocessor), ("reg", Ridge(alpha=1.0, random_state=42))])
all_results.append(
    evaluate_and_log_run(
        "6_Ridge_FE",
        pipe_ridge_fe,
        X_train_fe,
        y_train_fe,
        X_test_fe,
        y_test_fe,
        {"features": "domain_engineered", "alpha": 1.0},
    )
)

# Random Forest + Feature Engineering
pipe_rf_fe = Pipeline(
    [
        ("prep", fe_preprocessor),
        ("reg", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ]
)
all_results.append(
    evaluate_and_log_run(
        "7_Random_Forest_FE",
        pipe_rf_fe,
        X_train_fe,
        y_train_fe,
        X_test_fe,
        y_test_fe,
        {"features": "domain_engineered", "n_estimators": 100},
    )
)

## 8. Strategy 2: Cost-Sensitive Sample Weighting
Because older specimens are rare (only ~8% with `Rings > 15`), the MSE loss function underpenalizes errors on them.
We assign higher sample weights to specimens with `Rings > 12` during training ($2.0\times$ multiplier) to directly penalize underprediction on older abalones.

In [ ]:
# Define sample weights on training set
sample_weights_train = np.where(y_train_fe > 12, 2.0, 1.0)

# Weighted Ridge
pipe_ridge_w = Pipeline([("prep", fe_preprocessor), ("reg", Ridge(alpha=1.0, random_state=42))])
all_results.append(
    evaluate_and_log_run(
        "8_Ridge_FE_Weighted",
        pipe_ridge_w,
        X_train_fe,
        y_train_fe,
        X_test_fe,
        y_test_fe,
        params={"sample_weighting": "older_2x", "alpha": 1.0},
        fit_params={"reg__sample_weight": sample_weights_train},
    )
)

# Weighted Random Forest
pipe_rf_w = Pipeline(
    [
        ("prep", fe_preprocessor),
        ("reg", RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ]
)
all_results.append(
    evaluate_and_log_run(
        "9_Random_Forest_FE_Weighted",
        pipe_rf_w,
        X_train_fe,
        y_train_fe,
        X_test_fe,
        y_test_fe,
        params={"sample_weighting": "older_2x", "n_estimators": 100},
        fit_params={"reg__sample_weight": sample_weights_train},
    )
)

## 9. Comprehensive Model & Bias Comparison Table
We compare all models across general predictive accuracy (RMSE, MAE, R²) and targeted high-ring bias.

In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "Model": r["run_name"],
            "Test RMSE": round(r["test_rmse"], 3),
            "Test MAE": round(r["test_mae"], 3),
            "Test R2": round(r["test_r2"], 3),
            "Bias (>12 Rings)": round(r["bias_gt_12"], 2),
            "Bias (>15 Rings)": round(r["bias_gt_15"], 2),
        }
        for r in all_results
    ]
)
summary_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Baseline LR
t1 = f"1. Linear Reg Baseline\n(RMSE: {all_results[1]['test_rmse']:.2f}, Bias>15: {all_results[1]['bias_gt_15']:.2f})"
axes[0].scatter(y_test, all_results[1]["y_test_pred"], alpha=0.3, color="#2b5c8f")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_title(t1)
axes[0].set_xlabel("Actual Rings")
axes[0].set_ylabel("Predicted Rings")

# Random Forest + FE
t2 = f"2. Random Forest + FE\n(RMSE: {all_results[6]['test_rmse']:.2f}, Bias>15: {all_results[6]['bias_gt_15']:.2f})"
axes[1].scatter(y_test_fe, all_results[6]["y_test_pred"], alpha=0.3, color="#2ca02c")
axes[1].plot([y_test_fe.min(), y_test_fe.max()], [y_test_fe.min(), y_test_fe.max()], "r--", lw=2)
axes[1].set_title(t2)
axes[1].set_xlabel("Actual Rings")
axes[1].set_ylabel("Predicted Rings")

# Random Forest + FE + Sample Weighting
t3 = (
    "3. Random Forest + FE + Weighted\n"
    f"(RMSE: {all_results[8]['test_rmse']:.2f}, Bias>15: {all_results[8]['bias_gt_15']:.2f})"
)
axes[2].scatter(y_test_fe, all_results[8]["y_test_pred"], alpha=0.3, color="#d62728")
axes[2].plot([y_test_fe.min(), y_test_fe.max()], [y_test_fe.min(), y_test_fe.max()], "r--", lw=2)
axes[2].set_title(t3)
axes[2].set_xlabel("Actual Rings")
axes[2].set_ylabel("Predicted Rings")

plt.tight_layout()
plt.show()

## 10. Querying Runs from MLflow
We query the local MLflow experiment to verify all runs were tracked.

In [ ]:
runs = mlflow.search_runs(experiment_names=[experiment_name])
runs_table = runs[
    [
        "tags.mlflow.runName",
        "metrics.test_rmse",
        "metrics.test_r2",
        "metrics.bias_gt_12",
        "metrics.bias_gt_15",
        "start_time",
    ]
].sort_values(by="metrics.test_rmse")

runs_table.rename(columns={"tags.mlflow.runName": "Run Name"}, inplace=True)
runs_table

## 11. Key Findings & Discussion

### 1. Baseline Performance:
- The `DummyRegressor` establishes the baseline floor (Test RMSE = 3.412, $R^2 = -0.003$).
- `LinearRegression` on raw features achieves Test RMSE = 2.110 and $R^2 = 0.616$, providing an interpretable benchmark per course guidelines.

### 2. Strategy 1 (Feature Engineering) Impact:
- Adding **`shell_ratio`**, **`meat_ratio`**, and **`volume`** improves performance across all models:
  - `LinearRegression` Test RMSE drops from **2.110 $\rightarrow$ 2.076** ($R^2$ jumps from **0.616 $\rightarrow$ 0.629**).
  - `RandomForest` Test RMSE reaches the best overall score: **2.050** ($R^2 = 0.638$).
  - Bias on specimens with `Rings > 15` is reduced from **$-4.56 \rightarrow -4.08$ rings**.

### 3. Strategy 2 (Sample Weighting) Impact:
- Weighting older specimens ($2\times$ multiplier on `Rings > 12`) directly attacks the high-ring plateau:
  - Bias on `Rings > 12` drops to **$-2.07$ rings** (compared to $-2.82$ for LR Base and $-2.77$ for RF Base).
  - Bias on `Rings > 15` drops to **$-3.74$ rings** (compared to $-4.56$ initially, cutting underprediction error by over $0.8$ rings).
  - Overall accuracy remains high (Test RMSE = 2.102, $R^2 = 0.620$).

### 4. Trade-Off Summary:
- **Best overall RMSE & $R^2$**: `Random_Forest_FE` (RMSE = 2.050, $R^2 = 0.638$).
- **Best calibration on older specimens**: `Random_Forest_FE_Weighted` (cuts high-ring underprediction to -3.74).